# Extraction Gym — Benchmark Analysis

Visual analysis of benchmark results in `output/` powered by [`scripts/results.py`](scripts/results.py) and [`scripts/plots.py`](scripts/plots.py) (**Polars** + **plotnine**). All plots are automatically exported as PDFs to `plots/`.

- **Sanity Check**: verifies all optimal tree extractors agree on tree cost across every benchmark and raises a loud warning banner + diff table if not.
- **Part 1 — All-Extractor Overview**: summary table, combined pairwise speedup & win-rate heatmap, Dolan–Moré performance profile, and structural scaling curves (density/branching/cycles/depth).
- **Part 2 — Per-Suite Performance Breakdown**: suite × extractor geomean speedup & win-rate heatmap, faceted per-suite speedup distributions (IQR/p10–p90 intervals for large suites, raw points for small suites), and normalized cumulative wall-clock runtime by suite.
- **Part 3 — Pairwise Extractor Deep-Dive**: configurable pair (`E1` vs `E2`) and reachability thresholds, win/loss breakdown by suite, log–log runtime scatter, structural Spearman correlations, 2D speedup regime map, side-by-side reachability trends, and biggest wins/slowdowns.

In [ ]:
from pathlib import Path

from scripts import plots, results

plots.set_default_theme()

# --- Configuration ---
OUTPUT_DIR = Path("output")
PLOTS_DIR = Path("plots")

# Baseline extractor for multi-extractor ratio plots (Parts 1 & 2)
BASELINE = "faster-bottom-up-optimized"

# Extractor pair for the deep-dive plots (Part 3)
E1 = "a-star-interleaved"
E2 = "uniform-cost-search-optimized"

# Optional reachability thresholds
MIN_ENODES: int | None = None
MIN_ENODES_PCT: float | None = None
MIN_ECLASSES: int | None = None
MIN_ECLASSES_PCT: float | None = None

# Optional suite exclusion (e.g. ["dummy_examples"])
EXCLUDE_SUITES: list[str] = []

# Optional extractor exclusion (e.g. ["faster-greedy-dag", "faster-greedy-dag-optimized"])
EXCLUDE_EXTRACTORS: list[str] = [
    "faster-greedy-dag",
    "faster-greedy-dag-optimized",
    "bottom-up",
    "faster-bottom-up",
    "prio-queue",
    "uniform-cost-search-early-termination"
]

# Number of top wins / slowdowns to highlight in the outlier plot
TOP_N = 10

In [ ]:
raw_df = results.load_results_df(output_dir=OUTPUT_DIR, with_egraph_stats=True)
assert not raw_df.is_empty(), f"No benchmark JSON files found under {OUTPUT_DIR.resolve()}"

df = results.filter_results(
    raw_df,
    exclude_suites=EXCLUDE_SUITES,
    exclude_extractors=EXCLUDE_EXTRACTORS,
    min_enodes=MIN_ENODES,
    min_enodes_pct=MIN_ENODES_PCT,
    min_eclasses=MIN_ECLASSES,
    min_eclasses_pct=MIN_ECLASSES_PCT,
)

extractors = sorted(df["extractor"].unique().to_list())
if BASELINE not in extractors:
    BASELINE = extractors[0]
if E1 not in extractors:
    E1 = extractors[0]
if E2 not in extractors:
    E2 = extractors[min(1, len(extractors) - 1)]
PAIR_PREFIX = f"{E1}_vs_{E2}"

n_instances = df.select("name", "roots").n_unique()
print(f"Loaded {df.height:,} rows across {n_instances:,} benchmark instances and {len(extractors)} extractors:")
print("  Extractors:", ", ".join(extractors))

results.suite_counts_df(df)

In [ ]:
# Sanity check: all optimal tree extractors must agree on tree cost
results.check_tree_extractors(df, extractors=results.TREE_EXTRACTORS)

---
## Part 1 — All-Extractor Overview

In [ ]:
extractor_summary = results.extractor_summary_df(df)
extractor_order = extractor_summary["extractor"].to_list()
extractor_summary

In [ ]:
pairs_df = results.all_pairs_summary_df(df)

p = plots.plot_pairwise_heatmap(pairs_df, extractor_order=extractor_order)
plots.save_plot(p, "pairwise_heatmap", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_performance_profile(df)
plots.save_plot(p, "performance_profile", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_structure_regimes(df, BASELINE)
plots.save_plot(p, f"structure_regimes_vs_{BASELINE}", plots_dir=PLOTS_DIR)
p

---
## Part 2 — Per-Suite Performance Breakdown

Shows how each extractor performs on every case study (`herbie`, `tensat`, `babble`, `flexc`, etc.) without large suites drowning out smaller ones.

In [ ]:
p = plots.plot_suite_extractor_heatmap(df, BASELINE, extractor_order=extractor_order)
plots.save_plot(p, f"suite_heatmap_vs_{BASELINE}", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_suite_speedup_distributions(df, BASELINE, extractor_order=extractor_order)
plots.save_plot(p, f"suite_speedup_distributions_vs_{BASELINE}", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_suite_normalized_runtime(df, BASELINE, extractor_order=extractor_order)
plots.save_plot(p, f"suite_normalized_runtime_vs_{BASELINE}", plots_dir=PLOTS_DIR)
p

---
## Part 3 — Pairwise Extractor Deep-Dive (`E1` vs `E2`)

Change `E1`, `E2`, or the `MIN_ENODES*` / `MIN_ECLASSES*` filters in the top configuration cell to inspect any extractor pair and reachability slice.

In [ ]:
pair_df = results.pairwise_comparison_df(
    df,
    E1,
    E2,
    min_enodes=MIN_ENODES,
    min_enodes_pct=MIN_ENODES_PCT,
    min_eclasses=MIN_ECLASSES,
    min_eclasses_pct=MIN_ECLASSES_PCT,
)
print(results.summarize_pair(pair_df, E1, E2).format())

p = plots.plot_pair_suite_wins(pair_df, E1, E2)
plots.save_plot(p, f"{PAIR_PREFIX}_suite_wins", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_pair_scatter(pair_df, E1, E2)
plots.save_plot(p, f"{PAIR_PREFIX}_scatter", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_pair_feature_correlations(pair_df, E1, E2)
plots.save_plot(p, f"{PAIR_PREFIX}_feature_correlations", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_pair_regime_map(pair_df, E1, E2, x_col="enodes_per_eclass", y_col="avg_arity")
plots.save_plot(p, f"{PAIR_PREFIX}_regime_map", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_pair_speedup_vs_reachability(pair_df, E1, E2)
plots.save_plot(p, f"{PAIR_PREFIX}_reachability", plots_dir=PLOTS_DIR)
p

In [ ]:
p = plots.plot_pair_outliers(pair_df, E1, E2, top_n=TOP_N)
plots.save_plot(p, f"{PAIR_PREFIX}_outliers", plots_dir=PLOTS_DIR)
display(p)

results.pair_table_df(results.top_outliers_df(pair_df, E1, E2, top_n=TOP_N), E1, E2)